<a href="https://colab.research.google.com/github/rbrabusa/cmpt3830_alpaca/blob/main/cmpt_3830_streamlit_alpaca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GO Auto Loyalty-Driven Dealer Performance -- Streamlit App
## Team ALPACA -- Final Demo

This notebook builds a multi-page Streamlit dashboard for our Phase 2 analysis.
(%%writefile + Cloudflare tunnel in Colab).

**Pages:**
1. Home -- upload data and see overview KPIs
2. Customer Clustering -- segments, profiles, silhouette analysis
3. Dealer Performance -- benchmarking, loyalty impact, dealer map
4. Make-Specific Analysis -- per-brand clustering
5. Recommendations -- business insights with supporting charts
6. External Dashboard -- link to Power BI dashboard

## 1. Setup

In [ ]:
# install dependencies
!pip install streamlit pandas numpy scikit-learn matplotlib seaborn plotly folium streamlit-folium imbalanced-learn

In [ ]:
# install cloudflared for Colab tunneling
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## 2. Create Project Structure

We need to recreate the folder structure in Colab since it starts fresh each session.

In [ ]:
!mkdir -p goauto-app/{src,pages,models,data}
!touch goauto-app/src/__init__.py
print("Folders created.")

**Important:** Upload your `go_auto_cleaned_amended.csv` to the `goauto-app/data/` folder,
or use the sidebar uploader in the app.

In [ ]:
# if you have the csv in your Colab files or Google Drive, copy it here
# !cp /content/drive/MyDrive/go_auto_cleaned_amended.csv goauto-app/data/
# or just use the sidebar file uploader in the app

## 3. Source Files

In [ ]:
%%writefile goauto-app/src/config.py
RANDOM_STATE = 42
TEST_SIZE = 0.2
MODEL_PATH = "models/model.pkl"

# clustering features -- the 6 that passed professor's 0.50 silhouette threshold
CLUST_FEATURES = [
    'total_visits',
    'avg_cost',
    'avg_distance',
    'loyalty_ratio',
    'loyalty_treatment_ord',
    'scan_category_ord'
]

# ordinal mappings
LOYALTY_ORDER = {'Non-Loyal': 0, 'Partially Engaged': 1, 'Likely Loyal': 2}
SCAN_ORDER = {'Never Scanned': 0, 'Intermittent Scanner': 1, 'Always Scanned': 2}

# segment names from our Phase 2 analysis
SEGMENT_NAMES = {
    0: 'Non-Loyal Regulars',
    1: 'Engaged Power Users',
    2: 'Loyal Advocates',
    3: 'High-Spend Luxury',
    4: 'Distance Travelers'
}

# dealer coordinates (from Google Places)
DEALER_COORDS = {
    'Go Honda':                     (53.5443, -113.6414),
    'Norden Volkswagen':            (53.5988, -113.5760),
    'Toyota on the Trail':          (53.4895, -113.4963),
    'Kentwood Ford':                (53.5941, -113.4931),
    'Go Nissan South':              (53.4213, -113.4906),
    'Jaguar Land Rover Edmonton':   (53.5587, -113.6163),
    'Porsche Centre Edmonton':      (53.5404, -113.6361)
}

In [ ]:
%%writefile goauto-app/src/data.py
import pandas as pd
import numpy as np
import streamlit as st
from .config import LOYALTY_ORDER, SCAN_ORDER, CLUST_FEATURES

@st.cache_data
def load_and_clean(uploaded_file=None, path="data/go_auto_cleaned_amended.csv"):
    """Load the Phase 1 cleaned CSV and do ML-ready encoding."""
    if uploaded_file is not None:
        df = pd.read_csv(uploaded_file)
    else:
        df = pd.read_csv(path)

    # drop columns not needed
    drop_cols = [
        'service_date', 'sale_date', 'cost', 'mileage_outlier_flag',
        'distance_outlier_flag', 'cost_imputed', 'same_day_multi_service',
        'cost_min', 'cost_max', 'cost_normalized', 'year', 'sale_month', 'loyalty_card'
    ]
    drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=drop_cols)

    # ordinal encoding
    if 'loyalty_treatment' in df.columns:
        df['loyalty_treatment_ord'] = df['loyalty_treatment'].map(LOYALTY_ORDER)
        df = df.drop(columns=['loyalty_treatment'])

    if 'scan_category' in df.columns:
        df['scan_category_ord'] = df['scan_category'].map(SCAN_ORDER)
        df = df.drop(columns=['scan_category'])

    # frequency encoding for make/model
    if 'make' in df.columns:
        make_freq = df['make'].value_counts(normalize=True)
        df['make_name'] = df['make']  # keep original for later
        df['make_freq'] = df['make'].map(make_freq)
        df = df.drop(columns=['make'])

    if 'model' in df.columns:
        model_freq = df['model'].value_counts(normalize=True)
        df['model_freq'] = df['model'].map(model_freq)
        df = df.drop(columns=['model'])

    return df

@st.cache_data
def aggregate_customers(df):
    """Aggregate transactions to one row per customer (VIN)."""
    df_customer = df.groupby('vin').agg(
        total_visits=('loyalty_binary', 'count'),
        avg_cost=('cost_avg', 'mean'),
        avg_distance=('distance', 'mean'),
        avg_mileage=('mileage', 'mean'),
        vehicle_age=('vehicle_age', 'mean'),
        loyalty_ratio=('loyalty_ratio', 'first'),
        loyalty_binary=('loyalty_binary', 'first'),
        is_luxury=('is_luxury', 'max'),
        under_warranty=('under_warranty', 'max'),
        dealer_name=('dealer_name', 'first'),
        loyalty_treatment_ord=('loyalty_treatment_ord', 'first'),
        scan_category_ord=('scan_category_ord', 'first'),
        make_freq=('make_freq', 'first'),
        appointment_rate=('appointment', 'mean'),
        customer_pay_rate=('customer_pay', 'mean'),
        warranty_pay_rate=('warranty_pay', 'mean'),
    ).reset_index()

    # keep make name for make-specific analysis
    if 'make_name' in df.columns:
        vin_make = df.groupby('vin')['make_name'].first()
        df_customer['make_name'] = df_customer['vin'].map(vin_make)

    return df_customer

In [ ]:
%%writefile goauto-app/src/clustering.py
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from .config import CLUST_FEATURES, RANDOM_STATE, SEGMENT_NAMES

@st.cache_data
def run_clustering(df_customer, k=5):
    """Run KMeans clustering on customer-level data."""
    df_clust = df_customer[CLUST_FEATURES].copy()
    df_clust = df_clust.fillna(df_clust.median())

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_clust)

    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels)
    return labels, X_scaled, sil, scaler, kmeans

@st.cache_data
def find_best_k(df_customer, k_range=(2, 11)):
    """Test different k values and return scores."""
    df_clust = df_customer[CLUST_FEATURES].copy()
    df_clust = df_clust.fillna(df_clust.median())

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_clust)

    results = []
    for k in range(k_range[0], k_range[1]):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels)
        results.append({'k': k, 'silhouette': sil, 'inertia': km.inertia_})

    return pd.DataFrame(results)

@st.cache_data
def compare_algorithms(df_customer, k=5):
    """Compare KMeans vs GMM."""
    df_clust = df_customer[CLUST_FEATURES].copy().fillna(0)
    X_scaled = StandardScaler().fit_transform(df_clust)

    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km_labels = km.fit_predict(X_scaled)
    km_sil = silhouette_score(X_scaled, km_labels)

    gmm = GaussianMixture(n_components=k, random_state=RANDOM_STATE)
    gmm_labels = gmm.fit_predict(X_scaled)
    gmm_sil = silhouette_score(X_scaled, gmm_labels)

    return {'KMeans': km_sil, 'GMM': gmm_sil}

@st.cache_data
def cluster_profile(df_customer):
    """Generate cluster profiles."""
    profile = df_customer.groupby("cluster").agg(
        count=("cluster", "size"),
        avg_visits=("total_visits", "mean"),
        avg_cost=("avg_cost", "mean"),
        avg_distance=("avg_distance", "mean"),
        loyalty_ratio=("loyalty_ratio", "mean"),
        lt_ord=("loyalty_treatment_ord", "mean"),
        scan_ord=("scan_category_ord", "mean"),
        pct_luxury=("is_luxury", "mean"),
        pct_warranty=("under_warranty", "mean"),
    ).round(3)

    profile['pct'] = (profile['count'] / profile['count'].sum() * 100).round(1)
    return profile

## 4. App Pages

In [ ]:
%%writefile goauto-app/app.py
import streamlit as st
from src.data import load_and_clean, aggregate_customers

st.set_page_config(
    page_title="GO Auto -- Loyalty Analysis",
    page_icon="🚗",
    layout="wide"
)

st.title("🚗 GO Auto -- Loyalty-Driven Dealer Performance")
st.write("**Team ALPACA** | CMPT 3830 Machine Learning WIL")
st.write("Upload the cleaned dataset or place it in the data folder to begin.")

st.sidebar.header("Data Source")
uploaded = st.sidebar.file_uploader("Upload go_auto_cleaned_amended.csv", type=["csv"])

# try to load data
try:
    df = load_and_clean(uploaded_file=uploaded)
except Exception:
    st.info("Upload the CSV or place it in goauto-app/data/ to start.")
    st.stop()

df_customer = aggregate_customers(df)
st.session_state["df"] = df
st.session_state["df_customer"] = df_customer

# KPIs
st.subheader("Dataset Overview")
c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Transactions", f"{len(df):,}")
c2.metric("Unique Customers", f"{len(df_customer):,}")
c3.metric("Dealerships", df['dealer_name'].nunique())
c4.metric("Avg Loyalty Ratio", f"{df_customer['loyalty_ratio'].mean():.1%}")

c5, c6, c7, c8 = st.columns(4)
c5.metric("Avg Service Cost", f"${df_customer['avg_cost'].mean():.0f}")
c6.metric("Avg Visits/Customer", f"{df_customer['total_visits'].mean():.1f}")
c7.metric("Loyalty Members", f"{df_customer['loyalty_binary'].sum():,}")
c8.metric("Avg Distance (km)", f"{df_customer['avg_distance'].mean():.1f}")

st.subheader("Transaction Data Preview")
st.dataframe(df.head(20), use_container_width=True)

st.subheader("Customer-Level Data Preview")
st.dataframe(df_customer.head(20), use_container_width=True)

In [ ]:
%%writefile goauto-app/pages/1_Customer_Clustering.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from src.clustering import run_clustering, find_best_k, compare_algorithms, cluster_profile
from src.config import CLUST_FEATURES, SEGMENT_NAMES

st.title("Customer Clustering")

if "df_customer" not in st.session_state:
    st.warning("Load data on the Home page first.")
    st.stop()

df_customer = st.session_state["df_customer"].copy()

# sidebar controls
st.sidebar.header("Clustering Settings")
k = st.sidebar.slider("Number of clusters (k)", 2, 10, 5)

# run clustering
labels, X_scaled, sil, scaler, kmeans = run_clustering(df_customer, k=k)
df_customer["cluster"] = labels

# map segment names if k=5
if k == 5:
    df_customer["segment"] = df_customer["cluster"].map(SEGMENT_NAMES)
else:
    df_customer["segment"] = df_customer["cluster"].astype(str)

st.session_state["df_customer_clustered"] = df_customer

# metrics
st.subheader("Clustering Results")
c1, c2, c3 = st.columns(3)
c1.metric("Silhouette Score", f"{sil:.4f}")
c2.metric("Clusters", k)
c3.metric("Customers", f"{len(df_customer):,}")

# optimal k analysis
st.subheader("Optimal K Selection")
k_results = find_best_k(st.session_state["df_customer"])
best_k = k_results.loc[k_results['silhouette'].idxmax(), 'k']

col1, col2 = st.columns(2)
with col1:
    fig = px.line(k_results, x='k', y='inertia', markers=True, title='Elbow Method (Inertia)')
    fig.update_layout(xaxis_title='k', yaxis_title='Inertia')
    st.plotly_chart(fig, use_container_width=True)
with col2:
    fig = px.line(k_results, x='k', y='silhouette', markers=True, title='Silhouette Score vs k')
    fig.add_vline(x=best_k, line_dash="dash", line_color="red", annotation_text=f"Best k={int(best_k)}")
    fig.update_layout(xaxis_title='k', yaxis_title='Silhouette')
    st.plotly_chart(fig, use_container_width=True)

# algorithm comparison
st.subheader("Algorithm Comparison")
algo_results = compare_algorithms(st.session_state["df_customer"], k=k)
algo_df = pd.DataFrame(algo_results, index=["Silhouette"]).T
algo_df.columns = ["Silhouette Score"]
st.dataframe(algo_df, use_container_width=True)
winner = max(algo_results, key=algo_results.get)
st.success(f"Winner: **{winner}** (silhouette = {algo_results[winner]:.4f})")

# cluster profiles
st.subheader("Cluster Profiles")
profile = cluster_profile(df_customer)
st.dataframe(profile, use_container_width=True)

# cluster size
st.subheader("Cluster Size Distribution")
size_df = df_customer['segment'].value_counts().reset_index()
size_df.columns = ['Segment', 'Count']
fig = px.bar(size_df, x='Segment', y='Count', color='Segment', title='Customer Distribution Across Segments')
st.plotly_chart(fig, use_container_width=True)

# behavioral profile
st.subheader("Behavioral Profile by Cluster")
profile_cols = ['total_visits', 'avg_cost', 'avg_distance', 'loyalty_ratio']
profile_melted = df_customer.groupby('cluster')[profile_cols].mean().reset_index()
profile_melted = profile_melted.melt(id_vars='cluster', var_name='Feature', value_name='Value')
fig = px.bar(profile_melted, x='cluster', y='Value', color='Feature', barmode='group',
             title='Average Feature Values by Cluster')
st.plotly_chart(fig, use_container_width=True)

# loyalty engagement
st.subheader("Loyalty Engagement by Cluster")
fig = px.box(df_customer, x='segment', y='loyalty_ratio', color='segment',
             title='Loyalty Ratio Distribution by Segment')
st.plotly_chart(fig, use_container_width=True)

# scatter: distance vs cost
st.subheader("Distance vs Spending")
fig = px.scatter(df_customer, x='avg_distance', y='avg_cost', color='segment',
                 title='Customer Travel Distance vs Service Spending',
                 opacity=0.5)
st.plotly_chart(fig, use_container_width=True)

In [ ]:
%%writefile goauto-app/pages/2_Dealer_Performance.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
from streamlit_folium import st_folium
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from src.config import DEALER_COORDS

st.title("Dealer Performance")

if "df_customer" not in st.session_state:
    st.warning("Load data on the Home page first.")
    st.stop()

df_customer = st.session_state["df_customer"]

# dealer aggregation
dealer_perf = df_customer.groupby("dealer_name").agg(
    total_customers=("vin", "count"),
    avg_cost=("avg_cost", "mean"),
    avg_distance=("avg_distance", "mean"),
    avg_visits=("total_visits", "mean"),
    loyalty_rate=("loyalty_ratio", "mean"),
    pct_luxury=("is_luxury", "mean"),
).sort_values("loyalty_rate", ascending=False).round(2)

st.subheader("Dealer Summary Table")
st.dataframe(dealer_perf, use_container_width=True)

# dealer clustering
st.subheader("Dealer Clustering")
dealer_features = dealer_perf[["avg_cost", "avg_distance", "avg_visits", "loyalty_rate"]]
scaler_dealer = StandardScaler()
X_dealer = scaler_dealer.fit_transform(dealer_features)
kmeans_dealer = KMeans(n_clusters=3, random_state=42)
dealer_perf["dealer_cluster"] = kmeans_dealer.fit_predict(X_dealer)

fig = px.scatter(dealer_perf.reset_index(), x="loyalty_rate", y="avg_cost",
                 color="dealer_cluster", text="dealer_name",
                 size="total_customers", title="Dealer Loyalty Rate vs Revenue",
                 labels={"loyalty_rate": "Loyalty Rate", "avg_cost": "Avg Service Cost ($)"})
fig.update_traces(textposition="top center")
st.plotly_chart(fig, use_container_width=True)

# loyalty rate bar chart
st.subheader("Loyalty Engagement by Dealer")
loyalty_bar = dealer_perf.reset_index().sort_values("loyalty_rate")
fig = px.bar(loyalty_bar, x="dealer_name", y="loyalty_rate",
             title="Average Loyalty Participation by Dealer",
             labels={"dealer_name": "Dealer", "loyalty_rate": "Loyalty Rate"},
             color="loyalty_rate", color_continuous_scale="Greens")
st.plotly_chart(fig, use_container_width=True)

# cluster distribution by dealer
if "df_customer_clustered" in st.session_state:
    st.subheader("Customer Segment Distribution by Dealer")
    df_c = st.session_state["df_customer_clustered"]
    cross = pd.crosstab(df_c["dealer_name"], df_c["segment"], normalize="index").round(3)
    fig = px.bar(cross.reset_index().melt(id_vars="dealer_name"),
                 x="dealer_name", y="value", color="segment",
                 title="Customer Segment Mix by Dealer",
                 labels={"value": "Proportion", "dealer_name": "Dealer"})
    st.plotly_chart(fig, use_container_width=True)

# folium map
st.subheader("Dealer Location Map")

def loyalty_colour(rate):
    if rate > 0.25:
        return 'green'
    elif rate > 0.15:
        return 'orange'
    else:
        return 'red'

m = folium.Map(location=[53.55, -113.49], zoom_start=11)
for dealer in dealer_perf.index:
    if dealer in DEALER_COORDS:
        lat, lon = DEALER_COORDS[dealer]
        rate = dealer_perf.loc[dealer, 'loyalty_rate']
        customers = dealer_perf.loc[dealer, 'total_customers']
        colour = loyalty_colour(rate)
        folium.CircleMarker(
            location=[lat, lon],
            radius=max(5, customers / 500),
            popup=f"{dealer}<br>Loyalty: {rate:.1%}<br>Customers: {customers}",
            color=colour,
            fill=True,
            fill_opacity=0.7
        ).add_to(m)

st_folium(m, width=700, height=500)
st.caption("Green = high loyalty (>25%), Orange = medium (15-25%), Red = low (<15%)")

In [ ]:
%%writefile goauto-app/pages/3_Make_Specific_Analysis.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from src.config import CLUST_FEATURES, RANDOM_STATE

st.title("Make-Specific Clustering")

if "df_customer" not in st.session_state:
    st.warning("Load data on the Home page first.")
    st.stop()

df_customer = st.session_state["df_customer"]

if "make_name" not in df_customer.columns:
    st.warning("Make names not available in the data.")
    st.stop()

# top makes
n_top = st.sidebar.slider("Number of top makes", 3, 10, 7)
top_makes = df_customer['make_name'].value_counts().head(n_top).index.tolist()

# find best k per make
st.subheader("Optimal K per Make")
make_results = {}
for make in top_makes:
    subset = df_customer[df_customer['make_name'] == make]
    X_sub = subset[CLUST_FEATURES].fillna(0)
    X_sub_scaled = StandardScaler().fit_transform(X_sub)

    sils = {}
    for k_test in range(2, 8):
        km = KMeans(n_clusters=k_test, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X_sub_scaled)
        sils[k_test] = silhouette_score(X_sub_scaled, labels)

    best = max(sils, key=sils.get)
    make_results[make] = {'best_k': best, 'best_sil': sils[best], 'sils': sils, 'count': len(subset)}

# summary table
summary = pd.DataFrame({
    make: {'Best K': r['best_k'], 'Silhouette': round(r['best_sil'], 4), 'Customers': r['count']}
    for make, r in make_results.items()
}).T
st.dataframe(summary, use_container_width=True)

# silhouette plots per make
st.subheader("Silhouette Scores by K")
for make in top_makes:
    sils = make_results[make]['sils']
    fig = px.line(x=list(sils.keys()), y=list(sils.values()), markers=True,
                  title=f"{make} (best k={make_results[make]['best_k']}, n={make_results[make]['count']:,})",
                  labels={"x": "k", "y": "Silhouette"})
    st.plotly_chart(fig, use_container_width=True)

# per-make cluster profiles
st.subheader("Cluster Profiles per Make")
selected_make = st.selectbox("Select a make", top_makes)
subset = df_customer[df_customer['make_name'] == selected_make].copy()
X_sub = subset[CLUST_FEATURES].fillna(0)
X_sub_scaled = StandardScaler().fit_transform(X_sub)
best = make_results[selected_make]['best_k']

km = KMeans(n_clusters=best, random_state=RANDOM_STATE, n_init=10)
subset['make_cluster'] = km.fit_predict(X_sub_scaled)

profile = subset.groupby('make_cluster').agg(
    count=('make_cluster', 'size'),
    avg_visits=('total_visits', 'mean'),
    avg_cost=('avg_cost', 'mean'),
    avg_distance=('avg_distance', 'mean'),
    loyalty_ratio=('loyalty_ratio', 'mean')
).round(2)
st.dataframe(profile, use_container_width=True)

# loyalty breakdown
loyalty_dist = pd.crosstab(subset['make_cluster'], subset['loyalty_binary'], normalize='index').round(3)
loyalty_dist.columns = ['No Card', 'Has Card']
st.write("Loyalty card breakdown:")
st.dataframe(loyalty_dist, use_container_width=True)

In [ ]:
%%writefile goauto-app/pages/4_Recommendations.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.title("Business Recommendations")

if "df_customer_clustered" not in st.session_state:
    st.warning("Run clustering on the Clustering page first.")
    st.stop()

df = st.session_state["df_customer_clustered"]

st.markdown("""
### Key Findings from Our Analysis

These recommendations are based on the customer segmentation and dealer performance
analysis from our Phase 2 machine learning pipeline.
""")

# REC 1: loyalty engagement
st.subheader("1. Strengthen Loyalty Engagement")
st.write("Non-Loyal Regulars make up ~56% of all customers -- the biggest conversion opportunity.")
fig = px.box(df, x="segment", y="total_visits", color="segment",
             title="Service Visit Frequency by Segment")
st.plotly_chart(fig, use_container_width=True)

# REC 2: high-value segments
st.subheader("2. Target High-Value Customer Segments")
st.write("Engaged Power Users and Loyal Advocates visit more and spend consistently.")
fig = px.box(df, x="segment", y="avg_cost", color="segment",
             title="Service Spending by Segment")
st.plotly_chart(fig, use_container_width=True)

# REC 3: dealer performance
st.subheader("3. Optimize Dealer Performance")
st.write("Toyota on the Trail leads in loyalty adoption -- use as internal benchmark.")
dealer_loyalty = df.groupby("dealer_name")["loyalty_ratio"].mean().sort_values()
fig = px.bar(dealer_loyalty.reset_index(), x="dealer_name", y="loyalty_ratio",
             title="Average Loyalty Engagement by Dealer",
             labels={"dealer_name": "Dealer", "loyalty_ratio": "Loyalty Rate"},
             color="loyalty_ratio", color_continuous_scale="Greens")
st.plotly_chart(fig, use_container_width=True)

# REC 4: geographic targeting
st.subheader("4. Geographic Targeting")
st.write("Distance Travelers are willing to drive far -- consider targeted outreach.")
fig = px.histogram(df, x="avg_distance", nbins=30, color="segment",
                   title="Customer Travel Distance Distribution",
                   labels={"avg_distance": "Average Distance (km)"})
st.plotly_chart(fig, use_container_width=True)

# REC 5: luxury dealer paradox
st.subheader("5. Address the Luxury Dealer Paradox")
st.write("""
Premium dealerships (Jaguar Land Rover, Porsche) have the lowest loyalty enrollment
despite highest per-customer spend. There is significant revenue upside if loyalty
participation can be increased at these locations.
""")
dealer_summary = df.groupby("dealer_name").agg(
    loyalty_rate=("loyalty_ratio", "mean"),
    avg_spend=("avg_cost", "mean"),
    pct_luxury=("is_luxury", "mean")
).round(3)
fig = px.scatter(dealer_summary.reset_index(), x="loyalty_rate", y="avg_spend",
                 size="pct_luxury", text="dealer_name",
                 title="Loyalty Rate vs Avg Spend (bubble size = % luxury)",
                 labels={"loyalty_rate": "Loyalty Rate", "avg_spend": "Avg Service Cost ($)"})
fig.update_traces(textposition="top center")
st.plotly_chart(fig, use_container_width=True)

In [ ]:
%%writefile goauto-app/pages/5_External_Dashboard.py
import streamlit as st

st.title("External Dashboard")

st.write("""
Paste your Power BI or other dashboard link below to embed it alongside the Streamlit app.
""")

url = st.text_input("Dashboard URL")

if url:
    st.markdown(f"[Open Dashboard]({url})", unsafe_allow_html=True)
    st.markdown(
        f"""
        <iframe src="{url}"
        width="100%" height="800" style="border: none;">
        </iframe>
        """,
        unsafe_allow_html=True
    )
else:
    st.info("Enter a dashboard URL to display it here.")

## 5. Launch the App

Run the cells below to start the Streamlit server and open a Cloudflare tunnel.
The tunnel gives you a public URL you can share during the demo.

In [ ]:
# kill any old processes first
!pkill -f streamlit  || true
!pkill -f cloudflared || true
print("Old processes killed")

In [ ]:
%cd /content/goauto-app
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 \ --server.enableCORS false --server.enableXsrfProtection false \
  &> /content/streamlit.log &
print("Streamlit started.")

In [ ]:
# verify streamlit is running
!lsof -i:8501

In [ ]:
# create the public tunnel -- copy the .trycloudflare.com URL from the output
!cloudflared tunnel --url http://0.0.0.0:8501 --no-autoupdate

## 6. Cleanup (run when done)

In [ ]:
!pkill -f streamlit  || true
!pkill -f cloudflared || true
print("Processes stopped.")